# Importing all!

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "accretion").exists()),
    Path.cwd(),
)
SRC_PATH = PROJECT_ROOT / "src"
for path in (SRC_PATH,):
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

# IMPORT
# Configure notebook backends and autoreload before importing plotting modules.
%matplotlib widget
%load_ext autoreload
%autoreload 2

from accretion.ultimate_common import *

# Force-reload edited modules so the notebook picks up local changes immediately.
import accretion.ultimate_visualisation as _ultimate_visualisation
importlib.reload(_ultimate_visualisation)
import accretion.final_panel_colour_grid as _final_panel_colour_grid
importlib.reload(_final_panel_colour_grid)
import accretion.final_panels as _final_panels
importlib.reload(_final_panels)

# File/data helpers
from accretion.ultimate_file_organisers import (
    average_daily_measurements,
    get_daily_data,
    load_database_new,
    load_irtf_lxd_spectra,
    load_irtf_sxd_spectra,
    result_opener,
    result_saver,
)

# Physics helpers
from accretion.ultimate_physics import (
    accretion_model,
    flux_to_magnitude,
    log_flux_to_magnitude,
    wavelength_to_frequency,
    get_filter_wavelengths,
    planck_function,
    planck_model_custom,
    ccm89_extinction,
    temperature_profile_vectorized,
)

# Special-mode helper imports
from accretion.ultimate_helpers import (
    filter_days_for_red_excess,
    classify_filters_by_color,
    generate_ar_parameter_surface,
    optimize_mdot_for_av_rin,
)

# Core fitting imports
from accretion.ultimate_fitting import (
    _normalize_red_excess_fit_param,
    _safe_float,
    _resolve_red_excess_component,
    _coerce_wavelength_meters,
    _get_spectral_daily_data,
    ultimate_fitting_regularized,
)
from accretion.ultimate_fitting_types import (
    basic_fitting,
    allpams_fitting,
    AR_fitting,
    rede_fitting,
    spectral_fitting,
)

# Synthetic-data imports
from accretion.ultimate_synthetic_data import (
    generate_synthetic_data,
    save_synthetic_to_file,
    compare_original_synthetic,
)

# Visualisation imports
from accretion.ultimate_visualisation import (
    julian_to_calendar,
    add_gregorian_top_axis,
    get_model_predictions,
    plot_results_regularized,
    plot_parameter_evolution,
    compare_daily_params_evolution,
    create_color_plots_with_model,
    plot_unique_filter_with_model,
    create_residual_plots,
    comprehensive_results_visualization,
    plot_ar_surface,
    plot_red_excess_results,
    plot_spectral_fit_day,
    plot_spectral_fit_grid,
    visualize_spectral_results,
    visualize_results,
    plot_standard_results,
    example_visualization,
    _extract_regularized_series,
    _read_peter_results,
)

# Final-panel imports
from accretion.final_panel_colour_grid import plot_final_panel_colour_grid
from accretion.final_panels import plot_final_panels_0_to_4, plot_panel5_seds

# Fot estatics
%matplotlib widget


## Loading Data

In [ ]:
# Load your working photometry table

# Put your local data file in ../data/raw/ or update this path.
# For CSV files, use pd.read_csv. For the legacy whitespace format, use load_database_new.
data_path = Path('../data/raw/my_photometry.csv')
if not data_path.exists():
    raise FileNotFoundError(
        f"Put your photometry file at {data_path} or update data_path. "
        "See ../docs/data_format.md for required columns."
    )

df = pd.read_csv(data_path)
# df = load_database_new(data_path)

# Name any other filter. If you meet use unkown filters add them in both:
# src/accretion/ultimate_file_organisers.py
# src/accretion/ultimate_physics.py

for filter_name in ['I', 'J', 'H', 'K', 'L']:
    print(f"Number of {filter_name} filters", np.sum(np.where(df['Filter'] == filter_name, 1, 0)))


In [ ]:
## ── Load IRTF LXD + SXD spectra ──────────────────────────────────────────
# Usual spectral masks

lxd_mask_regions = [
    (1.6e-6,  2.45e-6),
    (2.45e-6, 2.9e-6),
    (4.15e-6, 4.6e-6),
]

sxd_mask_regions = [
    (0.68e-6, 0.74e-6),
    (1.3e-6,  1.41e-6),
    (1.82e-6, 1.95e-6),
    (2.45e-6, 2.9e-6),
]

JH_mask_regions = [
    (0.68e-6, 1.2e-6),
    (1.65e-6,  5.0e-6),
]

JHK_mask_regions = [
    (0.68e-6, 1.2e-6),
    (2.2e-6,  5.0e-6),
]

lxd_df = load_irtf_lxd_spectra(
    directory='IRTF',
    mask_regions=lxd_mask_regions,
    sample_stride=1,
)

sxd_df = load_irtf_sxd_spectra(
    directory='IRTF',
    mask_regions=sxd_mask_regions,
    sample_stride=1,
)

# Combine into one dataframe; the fitter groups by JD_day automatically
spectral_df = pd.concat([lxd_df, sxd_df], ignore_index=True)
spectral_df.sort_values(['JD', 'Lambda_m'], inplace=True)
spectral_df.reset_index(drop=True, inplace=True)

print('\nCombined samples per observing date:')
print(spectral_df.groupby(['DateCode', 'Filter'])['Lambda'].agg(['min', 'max', 'count']))

# JH-band subset of SXD: 1.2–1.6 µm; O2 telluric (1.3–1.4 µm) already removed by sxd_mask_regions
jh_df = sxd_df[(sxd_df['Lambda_m'] >= 1.2e-6) & (sxd_df['Lambda_m'] <= 1.6e-6)].copy()
jhk_df = sxd_df[(sxd_df['Lambda_m'] >= 1.2e-6) & (sxd_df['Lambda_m'] <= 2.2e-6)].copy()
print(f'\nJH subset: {len(jh_df)} samples across {jh_df["DateCode"].nunique()} dates')


# Fitting

## Spectral Fitting

In [ ]:
# Spectral fit – SXD only (0.8–2.5 µm, telluric bands masked)
sxd_results = spectral_fitting(
    sxd_df,
    lambda_reg=0,
    debug=False,
    save_dir='../results/generated/pictures/spectral_sxd',
)
result_saver(sxd_results, '../results/generated/datas/spectral_sxd')

# Spectral rede fit – SXD only (0.8–2.5 µm, telluric bands masked)
# sxd_rede_results = spectral_fitting(
#     sxd_df,
#     lambda_reg=0,
#     debug=False,
#     rede=True,
#     joint_refine=True,
#     max_blue_bb_fraction=0.10,
#     save_dir='../results/generated/pictures/spectral_sxd_rede',
# )
# result_saver(sxd_rede_results, '../results/generated/datas/spectral_sxd_rede')


In [ ]:
# Spectral fit – LXD only (2–5 µm, telluric bands masked)
lxd_results = spectral_fitting(
    lxd_df,
    lambda_reg=0,
    debug=False,
    save_dir='../results/generated/pictures/spectral_lxd',
)
result_saver(lxd_results, '../results/generated/datas/spectral_lxd')


In [ ]:
# # Spectral fit – combined SXD + LXD (full 0.8–5 µm coverage)
# combined_results = spectral_fitting(
#     spectral_df,
#     lambda_reg=0,
#     debug=False,
#     save_dir='../results/generated/pictures/spectral_combined',
# )
# result_saver(combined_results, '../results/generated/datas/spectral_combined')

# combined_rede_results = spectral_fitting(
#     spectral_df,
#     lambda_reg=0,
#     debug=False,
#     rede=True,
#     joint_refine=True,
#     max_blue_bb_fraction=0.10,
#     save_dir='../results/generated/pictures/spectral_combined_rede',
# )
# result_saver(combined_rede_results, '../results/generated/datas/spectral_combined_rede')


In [ ]:
# # Spectral fit – JH band only (1.2–1.6 µm from SXD, O2 telluric masked)
# jh_results = spectral_fitting(
#     jh_df,
#     lambda_reg=0,
#     debug=False,
#     save_dir='../results/generated/pictures/spectral_JH',
# )
# result_saver(jh_results, '../results/generated/datas/spectral_JH')


In [ ]:
# # Spectral fit – JHK band only (1.2–2.2 µm from SXD, O2 telluric masked)
# JHK = spectral_fitting(
#     jhk_df,
#     lambda_reg=0,
#     debug=False,
#     save_dir='../results/generated/pictures/spectral_JHK',
# )
# result_saver(JHK, '../results/generated/datas/spectral_JHK')


## Basic Fitting

In [ ]:
# # # Fitting JHK
basic_results = basic_fitting(df, debug=0)
result_saver(basic_results, '../results/generated/datas/basic_JHK')

In [ ]:
# # # Fitting JH
# basicJH_results = basic_fitting(df, required_filters=('J', 'H'), fit_filters=('J', 'H'))
# result_saver(basicJH_results, '../results/generated/datas/basic_JH')

## Extra Fitting

In [ ]:
# # Regularised

# reg1_results = basic_fitting(df, lambda_reg=1)
# synt_reg1_df = generate_synthetic_data(reg1_results, df, noise_level=0.1, random_seed=42)
# result_saver(reg1_results, '../results/generated/datas/reg_1')

# reg10_results = basic_fitting(df, lambda_reg=10)
# synt_reg10_df = generate_synthetic_data(reg10_results, df, noise_level=0.1, random_seed=42)
# result_saver(reg10_results, '../results/generated/datas/reg_10')

# reg100_results = basic_fitting(df, lambda_reg=100)
# synt_reg100_df = generate_synthetic_data(reg100_results, df, noise_level=0.1, random_seed=42)
# result_saver(reg100_results, '../results/generated/datas/reg_100')

In [ ]:
# # Red excess
# rede5IL_results_T = rede_fitting(df, required_filters=('I', 'J', 'H', 'K', 'L'), fit_filters=('I', 'J', 'H', 'K', 'L'), filename='rede5IL_T',  red_excess_fit_param='T_bb')
# rede5IL_results_R = rede_fitting(df, required_filters=('I', 'J', 'H', 'K', 'L'), fit_filters=('I', 'J', 'H', 'K', 'L'), filename='rede5IL_R', red_excess_fit_param='R_bb')

# result_saver(rede5IL_results_T, '../results/generated/datas/rede_5IL_T')
# result_saver(rede5IL_results_R, '../results/generated/datas/rede_5IL_R')

# rede4L_results_T = rede_fitting(df, required_filters=('J', 'H', 'K','L'), fit_filters=('J', 'H', 'K', 'L'), filename='rede4L_T', red_excess_fit_param='T_bb')
# rede4L_results_R = rede_fitting(df, required_filters=('J', 'H', 'K','L'), fit_filters=('J', 'H', 'K', 'L'), filename='rede4L_R', red_excess_fit_param='R_bb')

# result_saver(rede4L_results_T, '../results/generated/datas/rede_4L_T')
# result_saver(rede4L_results_R, '../results/generated/datas/rede_4L_R')

In [ ]:
# # Main console - AR mode
# AR_results = AR_fitting(df, 'AR_link')
# result_saver(AR_results, '../results/generated/datas/AR')

# Analysing

In [ ]:
# # JH

JH = result_opener(filepath='../results/generated/datas/basic_JH')

plot_results_regularized(JH, df, filename="JH_Evolution", end=True)
# generate_synthetic_data(JH, df, noise_level=0.1, random_seed=42)
compare_original_synthetic(df, JH, 
                           filter_req=['J', 'H'], 
                           filter_plot=['I', 'J', 'H', 'K', 'L+W1'], 
                           difference_view='filters',
                           filename="synthetic_vs_JH",
                           crosssection_day=2460848.5)


In [ ]:
# JHK

JHK = result_opener(filepath='../results/generated/datas/basic_JHK')


plot_results_regularized(JHK, df, filename="JHK_Evolution", end=True)
# generate_synthetic_data(JHK, df, noise_level=0.1, random_seed=42)
compare_original_synthetic(df, JHK, 
                           filter_req=['J', 'H', 'K'], 
                           filter_plot=['I', 'J', 'H', 'K', 'L+W1'], 
                           difference_view='filters',
                           filename="synthetic_vs_JHK",
                           crosssection_day=2460848.5)


In [ ]:
# # JHKL

# JHKL = result_opener(filepath='../results/generated/datas/rede_4L_T')

# plot_red_excess_results(JHKL, 'rede4L_T_evolution')
# compare_original_synthetic(df, JHKL, 
#                            filter_req=['J', 'H', 'K', 'L'], 
#                            filter_plot=['I','J', 'H', 'K', 'L'], 
#                            difference_view='filters',
#                            filename="red_excess/synthetic_vs_rede4L_T")

In [ ]:
# # IJHKL

# IJHKL = result_opener(filepath='../results/generated/datas/rede_5IL_T')

# plot_red_excess_results(IJHKL, 'rede5IL_T_evolution')
# compare_original_synthetic(df, IJHKL, 
#                            filter_req=['I','J', 'H', 'K', 'L'], 
#                            filter_plot=['I','J', 'H', 'K', 'L'], 
#                            difference_view='filters',
#                            filename="red_excess/synthetic_vs_rede5IL_T")

In [ ]:
# IRTF spectra – visualise all four spectral fit variants
for tag, save_dir in [
    ('spectral_sxd',      '../results/generated/datas/spectral_sxd')
    # ('spectral_sxd_rede', '../results/generated/datas/spectral_sxd_rede')
    ('spectral_lxd',      '../results/generated/datas/spectral_lxd')
    # ('spectral_combined', '../results/generated/datas/spectral_combined')
    # ('spectral_combined_rede', '../results/generated/datas/spectral_combined_rede'),
    # ('spectral_jh',       '../results/generated/datas/spectral_jh')
    # ('spectral_jhk',      '../results/generated/datas/spectral_jhk')
]:
    res = result_opener(filepath=f'../results/generated/datas/{tag}')
    visualize_spectral_results(res, df=spectral_df, save_dir="../results/generated/pictures/", show_range=[1, 4.2])


spectral_jhk = result_opener(filepath=f'../results/generated/datas/spectral_jhk')
spectral_jh = result_opener(filepath=f'../results/generated/datas/spectral_jh')

In [ ]:
# Compare saved daily-parameter files after you have generated results.
mod = 'spectroscopic'
results_dir = Path('../results/generated/datas')
result_files = list(results_dir.glob('*_daily_params.csv'))

if result_files:
    compare_daily_params_evolution(
        results_dir=str(results_dir),
        parameters=['Av', 'Mdot'],
        save_path=f'../results/generated/pictures/{mod}.pdf',
        model_mode=mod,
    )
else:
    print(f'No saved result bundles found in {results_dir}. Run a fitting cell and save results first.')

# modes: all, JH, JHK, photometric, spectroscopic, peter, Av_crosscheck


# Final comparison figures

In [ ]:
JH = result_opener(filepath='../results/generated/datas/basic_JH_newdf')
JHK = result_opener(filepath='../results/generated/datas/basic_JHK_newdf')
spectral_jh = result_opener(filepath=f'../results/generated/datas/spectral_jh')
spectral_jhk = result_opener(filepath=f'../results/generated/datas/spectral_jhk')
spectral_combined = result_opener(filepath=f'../results/generated/datas/spectral_combined_rede')
JHKL_redex = result_opener(filepath='../results/generated/datas/rede_4L_T')
# spectral_jhk = spectral_combined

default_params = {
        'M_star': 0.5 * M_sun,
        'R_star': 3.0 * R_sun,
        'R_in': 3.0 * R_sun,
        'R_out': 2 * constants.au,
        'distance': 700 * constants.parsec,
        'T_bb': RED_EXCESS_DEFAULT_T_BB,
        'R_bb': RED_EXCESS_DEFAULT_R_BB
    }

In [ ]:
# Final 5 panels for paper

# photo = JHK
# spect = spectral_jhk
# photometric_fit_filters = list(photo.get('fit_info', {}).get('fit_filters') or ['J', 'H', 'K'])

photo = JH
spect = spectral_jh
photometric_fit_filters = list(photo.get('fit_info', {}).get('fit_filters') or ['J', 'H'])


# SED dates. Accepts a single JD, list, tuple, or NumPy array.
specs = np.array([2460148, 2460565.5, 2460766.5, 2460781.5, 2460848])
brutes = np.array([2460816.5, 2460006.5, 2460766.5, 2460812.5])
sed_jd = 2460816.5

show_range = [0.78, 3.5]
panel5_save_dir = Path('../results/generated/pictures')
panel5_show = True

# Filter evolution and residual-panel filters.
# FILT = ('1.235', '1.662', '2.159')
FILT = ('J', 'H', 'K')
PLOT_PANEL0_FITS = False
Filt_diff = False
# date_range = [2459000, 2461200]
date_range = [None, None]

SHOW_PANEL3 = False
filter_plot_raw = ['J', 'H', 'K', 'L+W1']
FLUX_RESIDUAL_LOG_SCALE = False

fig_0_4, axes_0_4, panel_residuals = plot_final_panels_0_to_4(
    df,
    photo,
    spectral_results=spect,
    default_params=default_params,
    selected_filters=FILT,
    photometric_fit_filters=photometric_fit_filters,
    residual_filter_groups=filter_plot_raw,
    plot_panel0_fits=PLOT_PANEL0_FITS,
    filter_diff=Filt_diff,
    show_panel3=SHOW_PANEL3,
    date_range=date_range,
    flux_residual_log_scale=FLUX_RESIDUAL_LOG_SCALE,
    save_path='../results/generated/pictures/0-4_JH.pdf',
    show=True,
)

panel5_outputs = plot_panel5_seds(
    sed_jd,
    photo,
    spect,
    default_params=default_params,
    spectral_df=globals().get('spectral_df'),
    photometric_filters=panel_residuals['filters'],
    show_range=show_range,
    save_dir=panel5_save_dir,
    show=panel5_show,
)


In [ ]:
# Panel 6: Colour-colour and colour-magnitude diagrams for the selected filters, showing data and model evolution.
PANEL6_SHOW_FIT = False
plot = ("J,J-H", "J-H,H-K", "I,I-J", "K,H-K")

panel6_fig, panel6_matches, panel6_models = plot_final_panel_colour_grid(
    df,
    photo,
    tolerance=2.0,
    show_fit=PANEL6_SHOW_FIT,
    plot=plot,
    save_path='../results/generated/pictures/6_colour_grid.pdf',
    show=True,
)

for panel_name, panel_matches in panel6_matches.items():
    print(f'{panel_name}: {len(panel_matches)} matched raw-data points')
